# Fixed-Parameter Sensitivity / Identifiability Analysis

Compute-only notebook: runs `compute_sensitivity` (CasADi analytic derivatives) at
the canonical Bar-Even / Table IV literature parameters and saves all artifacts to
`results/fixed_estimation/`.  NO figures are produced here.

## 1. Imports and setup

In [1]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, 'src')

from kinetics_noor import EcoliCarbonKinetics
from sample_parameters import load_theta_init, build_theta_sources_df
from sentitivity import compute_sensitivity

RESULTS_DIR = Path('results/fixed_estimation')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('RESULTS_DIR:', RESULTS_DIR.resolve())


RESULTS_DIR: /Users/gabbi/Desktop/Repos/biosistemas/HybridKinetics-IIQ3733/results/fixed_estimation


## 2. Data import and build model

In [2]:
# Canonical analysis footing -- shared with first_estimation via
# src/utils.py: data-derived metabolite bounds + sorted conditions
# + measurement Q. Guarantees the sensitivities are computed on the same ground as
# first_estimation; only theta differs.
from utils import build_analysis_model, save_per_condition_sensitivity

# Same metabolite-bounds knob as first_estimation (log-space mean +/- N_STD*std).
N_STD = 3

model, valid_conds, input_enzyme, input_cell_needs, measurement_error = build_analysis_model('Data', n_std=N_STD)

print('Model built (data-derived bounds).')
print('  balanced_keys (%d):' % len(model.balanced_keys), model.balanced_keys)
print('  flux_keys     (%d):' % len(model.flux_keys),     model.flux_keys)
print('  conditions (%d, sorted):' % len(valid_conds), valid_conds)


Model built (data-derived bounds).
  balanced_keys (9): ['C_g6p', 'C_f6p', 'C_fbp', 'C_dhap', 'C_g3p', 'C_pgp', 'C_3pg', 'C_2pg', 'C_pep']
  flux_keys     (9): ['v_pts', 'v_pgi', 'v_pfkB', 'v_fbaA', 'v_tpiA', 'v_gapA', 'v_pgk', 'v_gpmA', 'v_eno']
  conditions (22, sorted): ['KO02', 'KO03', 'KO04', 'KO05', 'KO07', 'KO08', 'KO10', 'KO11', 'KO12', 'KO13', 'KO14', 'KO15', 'KO16', 'KO17', 'KO18', 'KO19', 'KO20', 'KO21', 'KO22', 'KO23', 'KO24', 'RF03']


## 3. Load canonical literature theta (seed 0)

In [3]:
theta_lit = load_theta_init('Data/theta_init_sampled.csv')

df_theta = build_theta_sources_df(theta_lit)
print('Loaded %d parameters (seed=0 canonical):' % len(theta_lit))
print(df_theta.to_string())


Loaded 37 parameters (seed=0 canonical):
                  value             source
param                                     
v_max_1       25.739000         literature
Ka1_1          1.000000         literature
Ka2_1          0.010000         literature
Ka3_1          1.000000         literature
K_g6p_1        0.500000         literature
Ks_g6p_pgi     1.018000         literature
Kp_f6p_pgi     0.078000         literature
kcat_f_2       4.545261       sampled_kcat
Ks_f6p_3       0.013000         literature
Ks_atp_3       0.020000         literature
Kp_fbp_3       0.140000         literature
Kp_adp_3       0.198355  sampled_km_linked
kcat_f_3       7.959356       sampled_kcat
Ks_fbp_4       0.240000         literature
Kp_g3p_4       0.044479  sampled_km_direct
Kp_dhap_4      0.533664  sampled_km_direct
kcat_f_4       0.950000         literature
kcat_f_5    1201.200000         literature
Ks_dhap_5      1.030000         literature
Kp_g3p_5       0.203263  sampled_km_direct
kcat_f_6     

## 4. Measurement error (Q)

In [4]:
measured_keys = [k for k in measurement_error if np.isfinite(measurement_error[k])]
print('Measured outputs: %d / %d' % (
    len(measured_keys), len(model.balanced_keys) + len(model.flux_keys)))
print(measured_keys)


Measured outputs: 14 / 18
['C_g6p', 'C_f6p', 'C_fbp', 'C_3pg', 'C_pep', 'v_pts', 'v_pgi', 'v_pfkB', 'v_fbaA', 'v_tpiA', 'v_gapA', 'v_pgk', 'v_gpmA', 'v_eno']


## 5. Inputs and conditions

In [5]:
# compute_sensitivity expects a list of {'condition': name} dicts
conditions = [{'condition': c} for c in valid_conds]
print('Conditions for compute_sensitivity (%d):' % len(conditions), valid_conds)


Conditions for compute_sensitivity (22): ['KO02', 'KO03', 'KO04', 'KO05', 'KO07', 'KO08', 'KO10', 'KO11', 'KO12', 'KO13', 'KO14', 'KO15', 'KO16', 'KO17', 'KO18', 'KO19', 'KO20', 'KO21', 'KO22', 'KO23', 'KO24', 'RF03']


## 6. Compute sensitivity (CasADi analytic)

In [ ]:
G_total, corr, diag = compute_sensitivity(
    model             = model,
    params_estimate   = theta_lit,
    input_enzyme      = input_enzyme,
    input_cell_needs  = input_cell_needs,
    conditions        = conditions,
    measurement_error = measurement_error,
)

print('G_total shape:', G_total.shape)
print('corr shape:',    corr.shape)
print('measured outputs (%d):' % len(diag['measured']), diag['measured'])


## 7. Save artifacts

In [ ]:
# correlation.csv (a-priori, relative-sensitivity FIM at literature theta)
corr_df = pd.DataFrame(corr, index=model.params_keys, columns=model.params_keys)
corr_df.to_csv(RESULTS_DIR / 'correlation.csv')
print('saved correlation.csv', corr_df.shape)

# per-condition absolute sensitivity CSVs + predictions/real on the SHARED footing
pred_df, real_df = save_per_condition_sensitivity(
    model, theta_lit, valid_conds, input_enzyme, input_cell_needs, RESULTS_DIR, data_dir='Data')
pred_df.to_csv(RESULTS_DIR / 'predictions.csv')
real_df.to_csv(RESULTS_DIR / 'real.csv')
print('saved sensitivity/ (%d conds) + predictions.csv %s + real.csv %s'
      % (len(valid_conds), pred_df.shape, real_df.shape))

# correlated_pairs.csv (|r| >= 0.9)
pairs = []
for i in range(len(model.params_keys)):
    for j in range(i + 1, len(model.params_keys)):
        r = corr[i, j]
        if np.isfinite(r) and abs(r) >= 0.9:
            pairs.append({'Parameter 1': model.params_keys[i],
                          'Parameter 2': model.params_keys[j],
                          'Correlation': r})
pairs_df = pd.DataFrame(pairs, columns=['Parameter 1', 'Parameter 2', 'Correlation'])
pairs_df.to_csv(RESULTS_DIR / 'correlated_pairs.csv', index=False)
print('saved correlated_pairs.csv (%d pairs with |r| >= 0.9)' % len(pairs_df))

# theta_lit.csv (value column for plots_fixed_estimation)
df_theta.to_csv(RESULTS_DIR / 'theta_lit.csv', index_label='param')
print('saved theta_lit.csv')

# manifest.json
manifest = {
    'seed':         0,
    'mode':         'literature; correlation analysis (compute_sensitivity, CasADi analytic)',
    'source':       'Data/theta_init_sampled.csv',
    'n_conditions': len(valid_conds),
    'conditions':   list(valid_conds),
    'n_measured':   len(measured_keys),
}
with open(RESULTS_DIR / 'manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=2)
print('saved manifest.json')

print()
print('=== results/fixed_estimation/ ===')
for p in sorted(RESULTS_DIR.iterdir()):
    print(' ', p.name)


saved correlation.csv (37, 37)
saved sensitivity/ (22 conds) + predictions.csv (22, 18) + real.csv (22, 18)
saved correlated_pairs.csv (537 pairs with |r| >= 0.9)
saved theta_lit.csv
saved manifest.json

=== results/fixed_estimation/ ===
  correlated_pairs.csv
  correlation.csv
  figures
  manifest.json
  predictions.csv
  real.csv
  sensitivity
  theta_lit.csv
